# Feature Engineering - F1 Podium Prediction

Membuat fitur untuk model prediksi podium sesuai rencana (Section 7).

## Fitur yang dibuat:
1. **Qualifying Features**: qualy_position, grid_effective, qual_delta, reached_q2/q3
2. **Driver Form**: rolling avg finish, points, podium_rate, win_rate, dnf_rate, grid_gain
3. **Constructor Form**: team points avg, team podium rate, team dnf rate
4. **Shifted Standings**: posisi dan points sebelum race
5. **Circuit History**: riwayat pembalap dan konstruktor di sirkuit yang sama
6. **Season Progress**: persentase musim telah berlalu
7. **Reliability Features**: mechanical_dnf_rate, crash_rate

**Semua rolling feature wajib menggunakan shift(1) untuk mencegah data leakage.**

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = '../data/'

In [ ]:
# Load semua data
results = pd.read_csv(DATA_PATH + 'results.csv', low_memory=False)
races = pd.read_csv(DATA_PATH + 'races.csv', low_memory=False)
qualifying = pd.read_csv(DATA_PATH + 'qualifying.csv', low_memory=False)
drivers = pd.read_csv(DATA_PATH + 'drivers.csv', low_memory=False)
constructors = pd.read_csv(DATA_PATH + 'constructors.csv', low_memory=False)
circuits = pd.read_csv(DATA_PATH + 'circuits.csv', low_memory=False)
driver_standings = pd.read_csv(DATA_PATH + 'driver_standings.csv', low_memory=False)
constructor_standings = pd.read_csv(DATA_PATH + 'constructor_standings.csv', low_memory=False)
sprint_results = pd.read_csv(DATA_PATH + 'sprint_results.csv', low_memory=False)
status = pd.read_csv(DATA_PATH + 'status.csv', low_memory=False)

print(f'results: {results.shape}')
print(f'races: {races.shape}')
print(f'qualifying: {qualifying.shape}')
print(f'drivers: {drivers.shape}')
print(f'constructors: {constructors.shape}')
print(f'circuits: {circuits.shape}')
print(f'driver_standings: {driver_standings.shape}')
print(f'constructor_standings: {constructor_standings.shape}')
print(f'sprint_results: {sprint_results.shape}')
print(f'status: {status.shape}')

## Data Cleaning

Konversi tipe data dan bersihkan nilai backslash-N.

In [ ]:
def clean_numeric(series):
    return pd.to_numeric(series.replace(r'\N', np.nan, regex=False), errors='coerce')

# Cleaning results
results['grid'] = clean_numeric(results['grid'])
results['positionOrder'] = clean_numeric(results['positionOrder'])
results['points'] = clean_numeric(results['points'])
results['laps'] = clean_numeric(results['laps'])
results['position'] = clean_numeric(results['position'])

# Cleaning races
races['date'] = pd.to_datetime(races['date'])

# Cleaning qualifying
qualifying = qualifying.replace(r'\N', np.nan, regex=False)
qualifying['position'] = clean_numeric(qualifying['position'])
qualifying = qualifying.rename(columns={'position': 'qualy_position'})

# Cleaning standings
driver_standings['position'] = clean_numeric(driver_standings['position'])
driver_standings['points'] = clean_numeric(driver_standings['points'])
constructor_standings['position'] = clean_numeric(constructor_standings['position'])
constructor_standings['points'] = clean_numeric(constructor_standings['points'])

# Cleaning sprint
sprint_results['grid'] = clean_numeric(sprint_results['grid'])
sprint_results['positionOrder'] = clean_numeric(sprint_results['positionOrder'])
sprint_results['position'] = clean_numeric(sprint_results['position'])
sprint_results['points'] = clean_numeric(sprint_results['points'])

print('Data cleaning selesai.')

## 1. Master Driver-Race Table

Buat tabel dasar: satu baris per driver per race.

In [ ]:
# Merge results + races
df = results.merge(
    races[['raceId', 'year', 'round', 'circuitId', 'date', 'name']],
    on='raceId', how='left'
)

# Merge drivers
df = df.merge(drivers[['driverId', 'driverRef', 'code', 'forename', 'surname']], on='driverId', how='left')

# Merge constructors
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left', suffixes=('', '_team'))
df = df.rename(columns={'name': 'race_name', 'name_team': 'team'})

# Sort by date
df = df.sort_values(['date', 'raceId', 'positionOrder']).reset_index(drop=True)

# Target
df['is_podium'] = (df['positionOrder'] <= 3).astype(int)

print(f'Master table: {df.shape}')
print(f'Tahun: {df["year"].min()} - {df["year"].max()}')
print(f'Total race unik: {df["raceId"].nunique()}')

### Koreksi Starting Grid

Grid 0 (pit lane start) -> treat sebagai posisi terakhir.

In [ ]:
# Grid efektif
field_size = df.groupby('raceId')['grid'].max()
df['field_size'] = df['raceId'].map(field_size)
df['grid_effective'] = df['grid'].fillna(df['field_size'] + 1)
df.loc[df['grid'] == 0, 'grid_effective'] = df['field_size'] + 1

print(f'Grid 0 count: {(df["grid"] == 0).sum()}')
print(f'Grid NaN count: {df["grid"].isna().sum()}')

## 2. Qualifying Features

- qualy_position
- reached_q2, reached_q3
- qual_gap_to_pole (delta time dari pole dalam detik)

In [ ]:
# Konversi waktu qualifying ke detik
def qual_time_to_seconds(time_str):
    if pd.isna(time_str):
        return np.nan
    try:
        parts = str(time_str).split(':')
        if len(parts) == 2:
            return int(parts[0]) * 60 + float(parts[1])
        return float(time_str)
    except:
        return np.nan

for col in ['q1', 'q2', 'q3']:
    qualifying[f'{col}_sec'] = qualifying[col].apply(qual_time_to_seconds)

# Best qualifying time per driver per race
qualifying['best_qual_sec'] = qualifying[['q1_sec', 'q2_sec', 'q3_sec']].min(axis=1)

# Reached Q2, Q3
qualifying['reached_q2'] = qualifying['q2'].notna().astype(int)
qualifying['reached_q3'] = qualifying['q3'].notna().astype(int)

# Gap to pole per sesi qualifying
pole_time = qualifying.groupby('raceId')['best_qual_sec'].transform('min')
qualifying['qual_gap_to_pole'] = qualifying['best_qual_sec'] - pole_time

qualifying[['raceId', 'driverId', 'qualy_position', 'best_qual_sec', 'qual_gap_to_pole', 'reached_q2', 'reached_q3']].head()

In [ ]:
# Merge qualifying features
qual_features = qualifying[[
    'raceId', 'driverId', 'qualy_position', 'best_qual_sec',
    'qual_gap_to_pole', 'reached_q2', 'reached_q3'
]]

df = df.merge(qual_features, on=['raceId', 'driverId'], how='left')

print(f'Setelah merge qualifying: {df.shape}')
print(f'qualy_position missing: {df["qualy_position"].isna().sum()}')

## 3. Driver Form Features

Rolling window features dengan shift(1) untuk cegah leakage.

- driver_finish_avg_3, _5, _10
- driver_points_avg_3, _5, _10
- driver_podium_rate_5, _10
- driver_win_rate_10
- driver_dnf_rate_5, _10
- driver_grid_gain_avg_5
- driver_prev_finish, driver_prev_qualy

In [ ]:
# Urutkan per driver berdasarkan waktu
df = df.sort_values(['driverId', 'date']).reset_index(drop=True)

# Fungsi rolling feature dengan shift(1)
def rolling_shifted(group, col, window, agg='mean'):
    """Hitung rolling dengan shift(1) per group"""
    if agg == 'mean':
        return group[col].shift(1).rolling(window, min_periods=1).mean()
    elif agg == 'std':
        return group[col].shift(1).rolling(window, min_periods=1).std()
    elif agg == 'max':
        return group[col].shift(1).rolling(window, min_periods=1).max()


# Driver finish position average (positionOrder closer to 1 = better)
df['driver_finish_avg_3'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'positionOrder', 3)
).reset_index(level=0, drop=True)

df['driver_finish_avg_5'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'positionOrder', 5)
).reset_index(level=0, drop=True)

df['driver_finish_avg_10'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'positionOrder', 10)
).reset_index(level=0, drop=True)

print('Driver finish averages selesai.')

In [ ]:
# Driver points average
df['driver_points_avg_3'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'points', 3)
).reset_index(level=0, drop=True)

df['driver_points_avg_5'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'points', 5)
).reset_index(level=0, drop=True)

df['driver_points_avg_10'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'points', 10)
).reset_index(level=0, drop=True)

print('Driver points averages selesai.')

In [ ]:
# Driver podium rate
df['driver_podium_rate_5'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'is_podium', 5)
).reset_index(level=0, drop=True)

df['driver_podium_rate_10'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'is_podium', 10)
).reset_index(level=0, drop=True)

# Driver win rate
df['is_win'] = (df['positionOrder'] == 1).astype(int)
df['driver_win_rate_10'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'is_win', 10)
).reset_index(level=0, drop=True)

print('Driver podium & win rates selesai.')

In [ ]:
# Driver DNF rate
# statusId = 1 berarti finis normal, lainnya = DNF atau masalah
df['is_finished'] = (df['statusId'] == 1).astype(int)
df['is_dnf'] = (~df['is_finished'].astype(bool)).astype(int)

df['driver_dnf_rate_5'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'is_dnf', 5)
).reset_index(level=0, drop=True)

df['driver_dnf_rate_10'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'is_dnf', 10)
).reset_index(level=0, drop=True)

print('Driver DNF rates selesai.')

In [ ]:
# Grid gain: positif = naik posisi dari start ke finish
df['grid_gain'] = df['grid_effective'] - df['positionOrder']
df['driver_grid_gain_avg_5'] = df.groupby('driverId').apply(
    lambda g: rolling_shifted(g, 'grid_gain', 5)
).reset_index(level=0, drop=True)

# Driver previous race finish (lag 1)
df['driver_prev_finish'] = df.groupby('driverId')['positionOrder'].shift(1)

# Driver previous race qualifying
df['driver_prev_qualy'] = df.groupby('driverId')['qualy_position'].shift(1)

print('Driver grid gain & prev features selesai.')

## 4. Constructor Form Features

- team_points_avg_5, _10
- team_best_finish_avg_5, _10
- team_podium_rate_10
- team_win_rate_10
- team_dnf_rate_5, _10
- constructor_prev_position, constructor_prev_points

In [ ]:
# Urutkan per konstruktor
df = df.sort_values(['constructorId', 'date']).reset_index(drop=True)

# Team points
df['team_points_avg_5'] = df.groupby('constructorId').apply(
    lambda g: rolling_shifted(g, 'points', 5)
).reset_index(level=0, drop=True)

df['team_points_avg_10'] = df.groupby('constructorId').apply(
    lambda g: rolling_shifted(g, 'points', 10)
).reset_index(level=0, drop=True)

# Team podium rate
df['team_podium_rate_10'] = df.groupby('constructorId').apply(
    lambda g: rolling_shifted(g, 'is_podium', 10)
).reset_index(level=0, drop=True)

# Team win rate
df['team_win_rate_10'] = df.groupby('constructorId').apply(
    lambda g: rolling_shifted(g, 'is_win', 10)
).reset_index(level=0, drop=True)

# Team DNF rate
df['team_dnf_rate_5'] = df.groupby('constructorId').apply(
    lambda g: rolling_shifted(g, 'is_dnf', 5)
).reset_index(level=0, drop=True)

df['team_dnf_rate_10'] = df.groupby('constructorId').apply(
    lambda g: rolling_shifted(g, 'is_dnf', 10)
).reset_index(level=0, drop=True)

# Constructor previous position and points
df['constructor_prev_points'] = df.groupby('constructorId')['points'].shift(1)
df['constructor_prev_position'] = df.groupby('constructorId')['positionOrder'].shift(1)

print('Constructor form features selesai.')

## 5. Shifted Standings

Posisi klasemen SEBELUM race (bukan setelah).
- driver_prev_standing_position
- driver_prev_standing_points
- driver_points_gap_to_leader
- constructor_prev_standing_position
- constructor_prev_standing_points

In [ ]:
# Driver standings: shift(1) per driver
driver_standings_sorted = driver_standings.sort_values(['driverId', 'raceId']).reset_index(drop=True)
driver_standings_sorted['prev_standing_pos'] = driver_standings_sorted.groupby('driverId')['position'].shift(1)
driver_standings_sorted['prev_standing_points'] = driver_standings_sorted.groupby('driverId')['points'].shift(1)

# Gap to leader: cari poin tertinggi per race, lalu hitung selisih
max_points_per_race_standings = driver_standings_sorted.groupby('raceId')['points'].transform('max')
driver_standings_sorted['points_gap_to_leader'] = max_points_per_race_standings - driver_standings_sorted['points']

# Shift juga gap to leader
driver_standings_sorted['prev_points_gap_to_leader'] = driver_standings_sorted.groupby('driverId')['points_gap_to_leader'].shift(1)

# Merge
df = df.merge(
    driver_standings_sorted[['raceId', 'driverId', 'prev_standing_pos', 'prev_standing_points', 'prev_points_gap_to_leader']],
    on=['raceId', 'driverId'],
    how='left'
)

# Constructor standings: shift(1) per constructor
constructor_standings_sorted = constructor_standings.sort_values(['constructorId', 'raceId']).reset_index(drop=True)
constructor_standings_sorted['prev_constructor_pos'] = constructor_standings_sorted.groupby('constructorId')['position'].shift(1)
constructor_standings_sorted['prev_constructor_points'] = constructor_standings_sorted.groupby('constructorId')['points'].shift(1)

# Max constructor points per race
max_cons_points = constructor_standings_sorted.groupby('raceId')['points'].transform('max')
constructor_standings_sorted['cons_points_gap_to_leader'] = max_cons_points - constructor_standings_sorted['points']
constructor_standings_sorted['prev_cons_gap_to_leader'] = constructor_standings_sorted.groupby('constructorId')['cons_points_gap_to_leader'].shift(1)

# Merge
df = df.merge(
    constructor_standings_sorted[
        ['raceId', 'constructorId', 'prev_constructor_pos', 'prev_constructor_points', 'prev_cons_gap_to_leader']
    ],
    on=['raceId', 'constructorId'],
    how='left'
)

# Fill NaN untuk race pertama musim
for col in ['prev_standing_pos', 'prev_standing_points', 'prev_constructor_pos', 'prev_constructor_points']:
    df[col] = df[col].fillna(0)

print('Shifted standings features selesai.')

## 6. Circuit History

Riwayat pembalap di sirkuit yang sama, hanya dari race sebelumnya.

In [ ]:
# Sort global by date
df = df.sort_values(['date', 'raceId', 'positionOrder']).reset_index(drop=True)

# Hitung statistik per driver + circuit
def create_circuit_history(df_data, group_cols, prefix):
    """
    Buat circuit history features per group.
    group_cols: ['driverId', 'circuitId'] atau ['constructorId', 'circuitId']
    """
    # Sort untuk akumulasi
    df_data = df_data.sort_values(group_cols + ['date']).reset_index(drop=True)
    
    # Rolling count of races at this circuit
    df_data[f'{prefix}_circuit_races'] = df_data.groupby(group_cols).cumcount()
    
    # Average finish at this circuit (expanding, shift untuk cegah leakage)
    df_data[f'{prefix}_circuit_avg_finish'] = (
        df_data.groupby(group_cols)['positionOrder']
        .apply(lambda x: x.shift(1).expanding(min_periods=1).mean())
    )
    
    # Podium rate at this circuit
    df_data[f'{prefix}_circuit_podium_rate'] = (
        df_data.groupby(group_cols)['is_podium']
        .apply(lambda x: x.shift(1).expanding(min_periods=1).mean())
    )
    
    # Best finish at this circuit
    df_data[f'{prefix}_circuit_best_finish'] = (
        df_data.groupby(group_cols)['positionOrder']
        .apply(lambda x: x.shift(1).expanding(min_periods=1).min())
    )
    
    return df_data


# Driver circuit history
df = create_circuit_history(df, ['driverId', 'circuitId'], 'driver')

# Constructor circuit history
df = create_circuit_history(df, ['constructorId', 'circuitId'], 'team')

# Fill NaN untuk first visit
for col in ['driver_circuit_avg_finish', 'driver_circuit_podium_rate', 'driver_circuit_best_finish',
            'team_circuit_avg_finish', 'team_circuit_podium_rate', 'team_circuit_best_finish']:
    df[col] = df[col].fillna(df[col].median() if col in ['driver_circuit_avg_finish', 'team_circuit_avg_finish'] else 0)

print('Circuit history features selesai.')

## 7. Sprint Features

Untuk sprint weekend: sprint_grid, sprint_finish, sprint_points.

In [ ]:
# Deteksi sprint weekend
sprint_races = sprint_results['raceId'].unique()
df['has_sprint'] = df['raceId'].isin(sprint_races).astype(int)

# Merge sprint results
sprint_features = sprint_results[['raceId', 'driverId', 'grid', 'positionOrder', 'points']].copy()
sprint_features = sprint_features.rename(columns={
    'grid': 'sprint_grid',
    'positionOrder': 'sprint_finish',
    'points': 'sprint_points'
})

df = df.merge(sprint_features, on=['raceId', 'driverId'], how='left')

# Isi NaN untuk non-sprint weekend
df['sprint_grid'] = df['sprint_grid'].fillna(0)
df['sprint_finish'] = df['sprint_finish'].fillna(0)
df['sprint_points'] = df['sprint_points'].fillna(0)

print(f'Sprint features selesai. Sprint weekends: {df["has_sprint"].sum()}')

## 8. Season Progress & Additional Features

In [ ]:
# Season progress
total_rounds = df.groupby('year')['round'].max().to_dict()
df['total_rounds_in_season'] = df['year'].map(total_rounds)
df['season_progress'] = df['round'] / df['total_rounds_in_season']

# Field size (jumlah pembalap yang start)
field_size_per_race = df.groupby('raceId')['grid_effective'].count()
df['race_field_size'] = df['raceId'].map(field_size_per_race)

print('Season progress selesai.')

## 9. Reliability Features

Kategorikan statusId menjadi: Finished, Mechanical DNF, Crash, Disqualified, Other.

In [ ]:
# Status kategorisasi
# status.csv: 1=Finished, 2-4=DSQ/DNF umum, 5-40=Crash, 41-80=Mechanical, 81+=Other
# Tapi kita mapping sederhana berdasarkan value umum F1 data

status_map = {
    1: 'finished',
    # Crash / Accident
    3: 'crash', 4: 'crash', 48: 'crash', 49: 'crash', 50: 'crash',
    51: 'crash', 52: 'crash', 53: 'crash', 54: 'crash', 73: 'crash',
    82: 'crash', 83: 'crash', 95: 'crash', 96: 'crash', 97: 'crash',
    98: 'crash', 99: 'crash', 100: 'crash', 101: 'crash', 102: 'crash',
    103: 'crash', 104: 'crash', 105: 'crash', 106: 'crash', 107: 'crash',
    108: 'crash', 109: 'crash', 110: 'crash', 111: 'crash', 112: 'crash',
    113: 'crash', 114: 'crash', 115: 'crash', 116: 'crash', 117: 'crash',
    # Mechanical DNF
    5: 'mechanical', 6: 'mechanical', 7: 'mechanical', 8: 'mechanical',
    9: 'mechanical', 10: 'mechanical', 11: 'mechanical', 12: 'mechanical',
    13: 'mechanical', 14: 'mechanical', 15: 'mechanical', 16: 'mechanical',
    17: 'mechanical', 18: 'mechanical', 19: 'mechanical', 20: 'mechanical',
    21: 'mechanical', 22: 'mechanical', 23: 'mechanical', 24: 'mechanical',
    25: 'mechanical', 26: 'mechanical', 27: 'mechanical', 28: 'mechanical',
    29: 'mechanical', 30: 'mechanical', 31: 'mechanical', 32: 'mechanical',
    33: 'mechanical', 34: 'mechanical', 35: 'mechanical', 36: 'mechanical',
    37: 'mechanical', 38: 'mechanical', 39: 'mechanical', 40: 'mechanical',
    41: 'mechanical', 42: 'mechanical', 43: 'mechanical', 44: 'mechanical',
    45: 'mechanical', 46: 'mechanical', 47: 'mechanical', 64: 'mechanical',
    75: 'mechanical', 76: 'mechanical', 77: 'mechanical', 78: 'mechanical',
    79: 'mechanical', 85: 'mechanical', 86: 'mechanical', 87: 'mechanical',
    88: 'mechanical', 89: 'mechanical', 90: 'mechanical', 91: 'mechanical',
    92: 'mechanical', 93: 'mechanical', 94: 'mechanical',
    # Disqualified
    2: 'disqualified', 55: 'disqualified', 56: 'disqualified', 57: 'disqualified',
    66: 'disqualified', 67: 'disqualified', 68: 'disqualified', 69: 'disqualified',
    70: 'disqualified', 71: 'disqualified', 72: 'disqualified',
}

# Map status
df['finish_status'] = df['statusId'].map(status_map).fillna('other')

# Binary indicators
df['is_mechanical_dnf'] = (df['finish_status'] == 'mechanical').astype(int)
df['is_crash'] = (df['finish_status'] == 'crash').astype(int)

# Rolling rate per driver
df['driver_mechanical_dnf_rate_5'] = df.groupby('driverId')['is_mechanical_dnf'].apply(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).reset_index(level=0, drop=True)

df['driver_crash_rate_5'] = df.groupby('driverId')['is_crash'].apply(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).reset_index(level=0, drop=True)

# Team rates
df['team_mechanical_dnf_rate_5'] = df.groupby('constructorId')['is_mechanical_dnf'].apply(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).reset_index(level=0, drop=True)

print('Reliability features selesai.')

## 10. Teammate Comparison

Bandingkan performa driver dengan rekan setim di race yang sama.

In [ ]:
# Cari teammate: driver lain di race yang sama dengan constructor yang sama
# Kita bisa lakukan dengan groupby raceId + constructorId

# Rata-rata finish position teammate per race
team_finish_mean = df.groupby(['raceId', 'constructorId'])['positionOrder'].transform('mean')
df['finish_gap_to_teammate_avg'] = df['positionOrder'] - team_finish_mean

# Rata-rata qualy position teammate
team_qualy_mean = df.groupby(['raceId', 'constructorId'])['qualy_position'].transform('mean')
df['qual_gap_to_teammate'] = df['qualy_position'] - team_qualy_mean

# Rolling qualifying win rate vs teammate
# Shifted: apakah di race sebelumnya qualy position lebih baik dari rata-rata tim?
df['qual_better_than_team_avg'] = (df['qualy_position'] < team_qualy_mean).astype(int)
df['qual_win_rate_vs_teammate_5'] = df.groupby('driverId')['qual_better_than_team_avg'].apply(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).reset_index(level=0, drop=True)

# Rolling race win rate vs teammate
df['finish_better_than_team_avg'] = (df['positionOrder'] < team_finish_mean).astype(int)
df['race_win_rate_vs_teammate_5'] = df.groupby('driverId')['finish_better_than_team_avg'].apply(
    lambda x: x.shift(1).rolling(5, min_periods=1).mean()
).reset_index(level=0, drop=True)

print('Teammate comparison features selesai.')

## 11. Final Dataset

Filter periode 2014+ dan simpan dataset final.

In [ ]:
# Filter era modern (2014+)
df_modern = df[df['year'] >= 2014].copy()

# Daftar fitur yang akan digunakan
feature_columns = [
    # Identitas
    'raceId', 'driverId', 'constructorId', 'year', 'round',
    'circuitId', 'date', 'race_name', 'driverRef', 'team',
    
    # Target
    'is_podium', 'positionOrder',
    
    # Qualifying
    'qualy_position', 'grid_effective', 'qual_gap_to_pole',
    'reached_q2', 'reached_q3',
    
    # Driver form
    'driver_finish_avg_3', 'driver_finish_avg_5', 'driver_finish_avg_10',
    'driver_points_avg_3', 'driver_points_avg_5', 'driver_points_avg_10',
    'driver_podium_rate_5', 'driver_podium_rate_10',
    'driver_win_rate_10',
    'driver_dnf_rate_5', 'driver_dnf_rate_10',
    'driver_grid_gain_avg_5',
    'driver_prev_finish', 'driver_prev_qualy',
    
    # Constructor form
    'team_points_avg_5', 'team_points_avg_10',
    'team_podium_rate_10', 'team_win_rate_10',
    'team_dnf_rate_5', 'team_dnf_rate_10',
    'constructor_prev_points', 'constructor_prev_position',
    
    # Standings
    'prev_standing_pos', 'prev_standing_points',
    'prev_points_gap_to_leader',
    'prev_constructor_pos', 'prev_constructor_points',
    'prev_cons_gap_to_leader',
    
    # Circuit history
    'driver_circuit_races', 'driver_circuit_avg_finish',
    'driver_circuit_podium_rate', 'driver_circuit_best_finish',
    'team_circuit_races', 'team_circuit_avg_finish',
    'team_circuit_podium_rate', 'team_circuit_best_finish',
    
    # Sprint
    'has_sprint', 'sprint_grid', 'sprint_finish', 'sprint_points',
    
    # Season
    'season_progress', 'race_field_size',
    
    # Reliability
    'driver_mechanical_dnf_rate_5', 'driver_crash_rate_5',
    'team_mechanical_dnf_rate_5',
    
    # Teammate comparison
    'qual_gap_to_teammate', 'finish_gap_to_teammate_avg',
    'qual_win_rate_vs_teammate_5', 'race_win_rate_vs_teammate_5',
]

# Cek kolom yang ada
available_features = [c for c in feature_columns if c in df_modern.columns]
missing_features = [c for c in feature_columns if c not in df_modern.columns]

print(f'Fitur tersedia: {len(available_features)} dari {len(feature_columns)}')
if missing_features:
    print(f'Fitur tidak ditemukan: {missing_features}')

df_final = df_modern[available_features].copy()
print(f'\nFinal dataset shape: {df_final.shape}')
print(f'Tahun: {df_final["year"].min()} - {df_final["year"].max()}')

In [ ]:
# Simpan dataset
df_final.to_parquet('../data/processed/model_dataset.parquet', index=False)
print('Dataset saved to data/processed/model_dataset.parquet')

In [ ]:
# Statistik dasar dataset final
print(f'Shape: {df_final.shape}')
print(f'Unique races: {df_final["raceId"].nunique()}')
print(f'Unique drivers: {df_final["driverId"].nunique()}')
print(f'Unique constructors: {df_final["constructorId"].nunique()}')
print(f'\nMissing values per kolom:')
missing_stats = df_final.isna().sum()
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)
display(missing_stats)